RAG PIPELINE -Data Ingestion to vector DB pipelines

In [1]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

d:\Ai\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory
!pip install pypdf
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")
print(all_pdf_documents)

Defaulting to user installation because normal site-packages is not writeable
Found 2 PDF files to process

Processing: Context.pdf


KeyboardInterrupt: 

In [ ]:

# ------------------------------------------------
# SPLIT DOCUMENTS INTO CHUNKS
# ------------------------------------------------
def chunk_splitting(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"\nSplit {len(documents)} documents into {len(split_docs)} chunks")

    # show example chunk
    if split_docs:
        print("\nExample Chunk:")
        print(split_docs[0].page_content[:200])
        print("\nMetadata:", split_docs[0].metadata)

    return split_docs


# ------------------------------------------------
# MAIN PIPELINE
# ------------------------------------------------
if __name__ == "__main__":

    pdf_directory = "../data"

    # Step 1: Load PDFs
    documents = process_all_pdfs(pdf_directory)

    # Step 2: Split into chunks
    chunked_data = chunk_splitting(documents)

    print(f"\nTotal chunks created: {len(chunked_data)}")

Found 2 PDF files to process

Processing: Context.pdf
  ✓ Loaded 23 pages

Processing: OOP notes.pdf
  ✓ Loaded 20 pages

Total documents loaded: 43

Split 43 documents into 83 chunks

Example Chunk:
Context
Engineering
Designing the systems that control what 
information reaches the model and how it 
maintains coherence.

Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2025-10-29T17:50:14+01:00', 'moddate': '2025-11-03T13:21:51+01:00', 'source': '..\\data\\pdf\\Context.pdf', 'total_pages': 23, 'page': 0, 'page_label': '1', 'source_file': 'Context.pdf', 'file_type': 'pdf'}

Total chunks created: 83


embedding and  vectorStoreDb

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

for Embedding first

In [ ]:

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()



    # Load model from HuggingFace
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            dim = self.model.get_sentence_embedding_dimension()
            print(f"Model loaded successfully. Embedding dimension: {dim}")

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise




    # Generate embeddings
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        
        if self.model is None:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )
        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings
    
embedding_manager=EmbeddingManager()
print(f"Embedding generate successfully :",embedding_manager)

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 988.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384
Embedding generate successfully : <__main__.EmbeddingManager object at 0x0000020EECB2CD70>


### Vector Store 


In [ ]:
import os
import chromadb
import uuid
import numpy as np
from typing import List, Any, Dict


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):

        self.collection_name = collection_name
        self.persist_directory = persist_directory

        self.collection = None
        self.client = None

        self._initialize_store()

    # Initialize ChromaDB and collection
    def _initialize_store(self):

        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF Documents Embeddings for RAG"
                }
            )

            print("Vector store initialized")
            print("Collection:", self.collection_name)
            print("Existing documents:", self.collection.count())

        except Exception as e:
            print("Error initializing vector store:", e)
            raise

    def AddDocuments(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} Documents to vector Store...")

        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["context_length"] = len(doc.page_content)

            metadatas.append(metadata)

            documents_texts.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                documents=documents_texts,
                metadatas=metadatas
            )

            print(f"Successfully added {len(documents)} documents to the vector store")
            print(f"Total Documents in Collection: {self.collection.count()}")

        except Exception as e:
            print("Error adding documents to vector store:", e)
            raise


# Initialize Vector Store
vectorstore = VectorStore()
print("Vector stored successfully:", vectorstore)


# Convert text into embeddings
texts = [doc.page_content for doc in chunked_data]

# Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Store embeddings in vector database
vectorstore.AddDocuments(chunked_data, embeddings)

Vector store initialized
Collection: pdf_documents
Existing documents: 0
Vector stored successfully: <__main__.VectorStore object at 0x0000020EECC03620>
Generating embeddings for 83 texts...


Batches: 100%|██████████| 3/3 [00:03<00:00,  1.32s/it]


Generated embeddings with shape: (83, 384)
Adding 83 Documents to vector Store...
Successfully added 83 documents to the vector store
Total Documents in Collection: 83


### retriver Pipeline From VectorStore


In [ ]:
from typing import List, Dict, Any


class RagRetriever:
    """Handles query-based retrieval from the vector store db"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Embedding manager used to generate query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def Retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            # search vector store
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results["documents"] and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):

                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "rank": i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
ragretrievers=RagRetriever(vectorstore,embedding_manager)
print(f"Retrived Successfully:",ragretrievers)

Retrived Successfully: <__main__.RagRetriever object at 0x0000020EEDE61FD0>


In [ ]:
ragretrievers.Retrieve("what is Context Engineering")

Retrieving documents for query: 'what is Context Engineering'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 56.43it/s]

Generated embeddings with shape: (1, 384)
Error during retrieval: type object 'VectorStore' has no attribute 'collection'


[]